# Exp 9 - IoV-based Vehicle Telemetry System using MQTT and Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Model an Internet-of-Vehicles telemetry stream using MQTT-style publish/subscribe messages.

MQTT is a lightweight publish/subscribe messaging protocol used in IoT settings. In this environment, no MQTT broker is running, so the notebook executes an in-memory publish/subscribe simulation while preserving MQTT concepts: topic, payload, publisher, and subscriber.

## Textbook Notes and Case Studies

### 1. Textbook Background

Internet of Vehicles telemetry systems collect data from vehicles and transmit it to edge, cloud, or monitoring systems. MQTT is a publish-subscribe messaging protocol commonly used for lightweight telemetry. In MQTT, clients publish messages to topics, and subscribers receive messages from topics they subscribe to through a broker.

The broker is the communication hub. Publishers and subscribers do not need to know each other's addresses. This decoupling is useful for telemetry because many vehicles can publish data while dashboards, analytics services, or alert engines subscribe independently.

### 2. Architecture Notes

```
Vehicle Sensors -> MQTT Publisher -> MQTT Broker -> Subscribers
       |                |                  |             |
       v                v                  v             v
Speed, GPS, Battery  Topic Payload    Topic Routing   Dashboard / IDS / Storage
```

The current notebook uses a Python fallback model because an MQTT broker and MQTT client package are not installed in the environment. The fallback preserves the topic-payload idea, but it is not a networked MQTT execution.

### 3. Important Concepts

Topic:

```
vehicle/{vehicle_id}/telemetry
```

Example payload fields:

```
timestamp, vehicle_id, speed, latitude, longitude, battery, status
```

Telemetry rate:

```
messages_per_second = number_of_messages / duration_seconds
```

Approximate payload throughput:

```
payload_throughput = total_payload_bytes / duration_seconds
```

### 4. Classroom Case Studies

Case Study A - Fleet Battery Monitoring:
Electric vehicles publish battery state and temperature. A monitoring service subscribes to telemetry topics and flags abnormal battery temperature.

Case Study B - Road Hazard Reporting:
Vehicles publish detected hazards to a topic. A roadside edge service subscribes and broadcasts localized alerts to nearby vehicles.

Case Study C - Predictive Maintenance:
Vehicle components publish vibration, temperature, and error-code data. Analytics services subscribe and detect early signs of failure.

### 5. Analysis Checklist

Report topic structure, payload fields, message count, message rate, and abnormal values. Clearly state whether the experiment used a real MQTT broker or a simulated broker-like fallback.

### 6. Source Notes

- MQTT Version 5.0 standard from OASIS: https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
- Python JSON module for payload modeling: https://docs.python.org/3/library/json.html


## Architecture

```text
Vehicle Sensor Data
  |-- speed
  |-- battery
  |-- sequence number
          |
          v
Publisher
  |-- topic: iov/vehicle/<id>/telemetry
  |-- JSON payload
          |
          v
MQTT Broker
  |-- not available here
  |-- replaced by in-memory list
          |
          v
Subscriber / Alert Filter
  |-- parse JSON
  |-- detect overspeed or low battery
```

## Formulas and Required Theory

Telemetry payload format:

\[
payload = \{vehicle, sequence, speed, battery\}
\]

Alert rules used in this notebook:

\[
\text{overspeed} =
\begin{cases}
1, & speed > 80\text{ km/h}\\
0, & otherwise
\end{cases}
\]

\[
\text{low battery} =
\begin{cases}
1, & battery < 30\%\\
0, & otherwise
\end{cases}
\]

## In-Lab Method

1. Create telemetry messages as JSON.
2. Assign each message to a topic.
3. Simulate publishing messages.
4. Simulate subscriber-side parsing.
5. Print messages and verify payload structure.

In [1]:
import json
import random
import time

print("EXP 9 - IN-LAB MQTT TELEMETRY")
print("No MQTT broker is available here, so this executes an in-memory publish/subscribe simulation.")
rng = random.Random(341409)
messages = []
for i in range(5):
    payload = {"vehicle": "KL-V2X-01", "seq": i, "speed_kmph": round(rng.uniform(20, 65), 1), "battery_pct": round(rng.uniform(55, 94), 1)}
    topic = "iov/vehicle/KL-V2X-01/telemetry"
    messages.append((topic, json.dumps(payload)))
for topic, payload in messages:
    print(topic, payload)

EXP 9 - IN-LAB MQTT TELEMETRY
No MQTT broker is available here, so this executes an in-memory publish/subscribe simulation.
iov/vehicle/KL-V2X-01/telemetry {"vehicle": "KL-V2X-01", "seq": 0, "speed_kmph": 24.5, "battery_pct": 65.0}
iov/vehicle/KL-V2X-01/telemetry {"vehicle": "KL-V2X-01", "seq": 1, "speed_kmph": 63.0, "battery_pct": 62.6}
iov/vehicle/KL-V2X-01/telemetry {"vehicle": "KL-V2X-01", "seq": 2, "speed_kmph": 40.7, "battery_pct": 80.7}
iov/vehicle/KL-V2X-01/telemetry {"vehicle": "KL-V2X-01", "seq": 3, "speed_kmph": 64.3, "battery_pct": 59.7}
iov/vehicle/KL-V2X-01/telemetry {"vehicle": "KL-V2X-01", "seq": 4, "speed_kmph": 36.7, "battery_pct": 85.4}


## Post-Lab Method

The post-lab cell applies alert rules to incoming telemetry. It classifies each vehicle state as normal, overspeed, low battery, or both.

In [2]:
import json

print("EXP 9 - POST-LAB ALERT FILTER")
raw_messages = [
    '{"vehicle":"V1","speed_kmph":42,"battery_pct":80}',
    '{"vehicle":"V2","speed_kmph":88,"battery_pct":48}',
    '{"vehicle":"V3","speed_kmph":35,"battery_pct":22}',
]
for raw in raw_messages:
    msg = json.loads(raw)
    alerts = []
    if msg["speed_kmph"] > 80:
        alerts.append("overspeed")
    if msg["battery_pct"] < 30:
        alerts.append("low battery")
    print(msg["vehicle"], "alerts=", alerts or ["normal"])

EXP 9 - POST-LAB ALERT FILTER
V1 alerts= ['normal']
V2 alerts= ['overspeed']
V3 alerts= ['low battery']


## What to Write in the Lab Record

- Include sample MQTT topic names.
- Include sample JSON payloads.
- Explain why publish/subscribe decouples vehicles from monitoring dashboards.
- State that a real deployment requires an MQTT broker and network security controls.

## References

- MQTT Version 5.0 OASIS Standard: https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html
- MQTT project overview: https://mqtt.org/
- Python `json` module documentation: https://docs.python.org/3/library/json.html